# Fraud Detection in Financial Transactions

This notebook is a reproducible workflow for EDA, imbalance handling, supervised fraud classification, Isolation Forest, Autoencoder anomaly detection, threshold selection, and alert generation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, precision_recall_curve
sys.path.append(str(Path.cwd().parents[0]))
from src.data_preprocessing import load_transactions, build_preprocessor, split_features_target
from src.supervised_models import train_models, evaluate_binary_model, oversample_smote, find_recall_threshold
from src.anomaly_detection import fit_isolation_forest, isolation_scores, train_autoencoder, reconstruction_scores
from src.alerting import build_alerts

In [ ]:
df = load_transactions('data/raw/transactions.csv')
display(df.head())
display(df.dtypes)
print(df.shape)

## EDA

In [ ]:
print(df.isna().mean().sort_values(ascending=False).head(15))
if 'is_fraud' in df.columns:
    print(df['is_fraud'].value_counts(normalize=True))
    sns.countplot(data=df, x='is_fraud')
    plt.title('Fraud Class Distribution')
    plt.show()

## Preprocess

In [ ]:
X, y = split_features_target(df, 'is_fraud')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocessor = build_preprocessor(X_train)
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
print(X_train_t.shape, X_test_t.shape)

## Supervised models and imbalance handling

In [ ]:
models = train_models(X_train_t, y_train.to_numpy())
metrics = {name: evaluate_binary_model(model, X_test_t, y_test.to_numpy()) for name, model in models.items()}
pd.DataFrame(metrics).T

In [ ]:
X_smote, y_smote = oversample_smote(X_train_t, y_train.to_numpy())
print('Before SMOTE:', np.bincount(y_train))
print('After SMOTE:', np.bincount(y_smote))

## Isolation Forest

In [ ]:
iso = fit_isolation_forest(X_train_t, contamination=max(0.001, float(y_train.mean())))
iso_scores = isolation_scores(iso, X_test_t)
print('Isolation Forest score range:', iso_scores.min(), iso_scores.max())

## Autoencoder

In [ ]:
normal_train = X_train_t[y_train.to_numpy() == 0]
ae = train_autoencoder(normal_train, epochs=30, batch_size=256)
ae_scores = reconstruction_scores(ae, X_test_t)
print('Autoencoder score range:', ae_scores.min(), ae_scores.max())

## Threshold tuning and alerts

In [ ]:
rf = models['random_forest']
rf_scores = rf.predict_proba(X_test_t)[:, 1]
threshold = find_recall_threshold(y_test.to_numpy(), rf_scores, target_recall=0.90)
print('Selected threshold:', threshold)
print(classification_report(y_test, (rf_scores >= threshold).astype(int), digits=4))
print('ROC-AUC:', roc_auc_score(y_test, rf_scores))
print('PR-AUC:', average_precision_score(y_test, rf_scores))

In [ ]:
test_display = X_test.reset_index(drop=True).copy()
test_display['is_fraud'] = y_test.reset_index(drop=True)
alerts = build_alerts(test_display, rf_scores, score_column='fraud_score')
Path('data/processed').mkdir(parents=True, exist_ok=True)
alerts.to_csv('data/processed/scored_transactions.csv', index=False)
display(alerts.head(20))